<a href="https://colab.research.google.com/github/vinileodido/MVP_EngDados_PUC/blob/main/StreamlitVersions/mvp_anatelsmp_streamlitapp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install streamlit pyngrok pandas altair matplotlib requests -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 2.5 MB/s eta 0:00:00


In [2]:
from pyngrok import ngrok
import getpass

# Obter token do ngrok (cadastre-se em https://ngrok.com/)
print("🔑 Para usar este dashboard, você precisa de um token do ngrok.")
print("1. Acesse: https://ngrok.com/")
print("2. Faça cadastro gratuito")
print("3. Copie seu token de autenticação")
print("4. Cole abaixo:")

ngrok_token = getpass.getpass("Token do ngrok: ")
ngrok.set_auth_token(ngrok_token)

🔑 Para usar este dashboard, você precisa de um token do ngrok.
1. Acesse: https://ngrok.com/
2. Faça cadastro gratuito
3. Copie seu token de autenticação
4. Cole abaixo:
Token do ngrok: ··········


In [3]:
#@title App Streamlit
%%writefile streamlit_app.py

import streamlit as st
import pandas as pd
import numpy as np
import altair as alt
import matplotlib.pyplot as plt
import requests
import io

# Configuração da página
st.set_page_config(
    page_title="Dashboard ANATEL",
    layout="wide",
    initial_sidebar_state="expanded"
)

# CSS personalizado
st.markdown("""
<style>
    .main-header {
        font-size: 2.5rem;
        color: #1f4e79;
        text-align: center;
        margin-bottom: 2rem;
        border-bottom: 3px solid #1f4e79;
        padding-bottom: 1rem;
    }
    .metric-card {
        background-color: #f0f2f6;
        padding: 1rem;
        border-radius: 0.5rem;
        border-left: 4px solid #1f4e79;
    }
    .sidebar .sidebar-content {
        background-color: #f8f9fa;
    }
</style>
""", unsafe_allow_html=True)

# Função para carregar dados com cache
@st.cache_data(ttl=3600)  # Cache por 1 hora
def load_data():
    """Carrega dados das ERBs da ANATEL"""
    try:
        base_url = "https://github.com/vinileodido/MVP_EngDados_PUC/raw/refs/heads/main/Datasets/"

        # Carregar dados Brasil
        df_br = pd.read_csv(
            f"{base_url}vw_erbs_br.csv",
            names=['Operadora', '2G', '3G', '4G', '5G']
        )

        # Carregar dados por UF
        df_uf = pd.read_csv(
            f"{base_url}vw_erbs_uf.csv",
            names=['UF', 'COD_UF', 'COD_AREA', 'Operadora', '2G', '3G', '4G', '5G']
        )

        # Carregar dados por cidade
        df_cidade = pd.read_csv(
            f"{base_url}vw_erbs_cid.csv",
            names=['UF', 'COD_UF', 'COD_AREA', 'COD_IBGE', 'Cidade', 'CAPITAL', 'Operadora', '2G', '3G', '4G', '5G']
        )

        # Converter colunas numéricas
        for df in [df_br, df_uf, df_cidade]:
            for col in ['2G', '3G', '4G', '5G']:
                if col in df.columns:
                    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

        return df_br, df_uf, df_cidade

    except Exception as e:
        st.error(f"Erro ao carregar dados: {e}")
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

# Função para criar gráficos
def create_bar_chart(df, title):
    """Cria gráfico de barras com Altair"""
    df_melted = df.melt(
        id_vars=['Operadora'],
        value_vars=['2G', '3G', '4G', '5G'],
        var_name='Tecnologia',
        value_name='Quantidade'
    )

    chart = alt.Chart(df_melted).mark_bar().encode(
        x=alt.X('Operadora:N', title='Operadora'),
        y=alt.Y('Quantidade:Q', title='Quantidade de ERBs'),
        color=alt.Color('Tecnologia:N',
            scale=alt.Scale(
                domain=['2G', '3G', '4G', '5G'],
                range=['#FF6B35', '#004E89', '#1A8F5A', '#8E44AD']
            )
        ),
        tooltip=['Operadora', 'Tecnologia', 'Quantidade']
    ).properties(
        width='container',
        height=400,
        title=title
    )

    return chart

def create_pie_chart(values, labels, title):
    """Cria gráfico de pizza com matplotlib"""
    fig, ax = plt.subplots(figsize=(10, 8))
    colors = ['#FF6B35', '#004E89', '#1A8F5A', '#8E44AD']

    total = sum(values)
    percentages = [v/total*100 for v in values]
    labels_with_values = [f'{label}\n{val:,.0f} ({pct:.1f}%)'
                         for label, val, pct in zip(labels, values, percentages)]

    wedges, texts, autotexts = ax.pie(
        values,
        labels=labels_with_values,
        colors=colors,
        autopct='',
        startangle=90,
        textprops={'fontsize': 10, 'weight': 'bold'}
    )

    ax.set_title(title, fontsize=16, fontweight='bold', pad=20)
    return fig

# INÍCIO DA APLICAÇÃO
st.markdown('<h1 class="main-header">🏢 Dashboard ANATEL - Telecomunicações Brasil</h1>', unsafe_allow_html=True)

# Sidebar
st.sidebar.title("🧭 Navegação")
st.sidebar.markdown("---")

# Carregar dados
with st.spinner("🔄 Carregando dados da ANATEL..."):
    df_br, df_uf, df_cidade = load_data()

if df_br.empty:
    st.error("❌ Não foi possível carregar os dados. Verifique sua conexão com a internet.")
    st.stop()

# Seleção de página
page = st.sidebar.selectbox(
    "Selecione a análise:",
    ["📊 Visão Nacional", "🗺️ Por Estado", "🏙️ Por Cidade", "📈 Comparações"]
)

# Informações na sidebar
with st.sidebar.expander("ℹ️ Sobre os Dados"):
    st.write("""
    **Fonte:** ANATEL (Agência Nacional de Telecomunicações)

    **ERBs:** Estações Rádio Base

    **Tecnologias:**
    - 2G: GSM
    - 3G: WCDMA
    - 4G: LTE
    - 5G: NR (New Radio)

    **Última atualização:** Dados mais recentes disponíveis
    """)

# PÁGINA PRINCIPAL - VISÃO NACIONAL
if page == "📊 Visão Nacional":
    st.header("📊 Análise Nacional de ERBs")

    # Calcular métricas
    total_2g = df_br['2G'].sum()
    total_3g = df_br['3G'].sum()
    total_4g = df_br['4G'].sum()
    total_5g = df_br['5G'].sum()
    total_geral = total_2g + total_3g + total_4g + total_5g

    # Exibir métricas principais
    col1, col2, col3, col4, col5 = st.columns(5)

    with col1:
        st.metric("🏢 Total ERBs", f"{total_geral:,.0f}".replace(",", "."))
    with col2:
        st.metric("📱 2G", f"{total_2g:,.0f}".replace(",", "."))
    with col3:
        st.metric("📶 3G", f"{total_3g:,.0f}".replace(",", "."))
    with col4:
        st.metric("📡 4G", f"{total_4g:,.0f}".replace(",", "."))
    with col5:
        st.metric("🚀 5G", f"{total_5g:,.0f}".replace(",", "."))

    st.markdown("---")

    # Tabs para diferentes visualizações
    tab1, tab2, tab3 = st.tabs(["📊 Dados por Operadora", "📈 Gráfico de Barras", "🥧 Distribuição por Tecnologia"])

    with tab1:
        st.subheader("Dados Detalhados por Operadora")

        # Adicionar coluna de total
        df_display = df_br.copy()
        df_display['Total'] = df_display['2G'] + df_display['3G'] + df_display['4G'] + df_display['5G']
        df_display = df_display.sort_values('Total', ascending=False)

        # Aplicar formatação
        df_styled = df_display.style.format({
            '2G': '{:,.0f}',
            '3G': '{:,.0f}',
            '4G': '{:,.0f}',
            '5G': '{:,.0f}',
            'Total': '{:,.0f}'
        })

        st.dataframe(df_styled, use_container_width=True, hide_index=True)

    with tab2:
        st.subheader("Distribuição de ERBs por Operadora e Tecnologia")
        chart = create_bar_chart(df_br, "ERBs por Operadora e Tecnologia")
        st.altair_chart(chart, use_container_width=True)

    with tab3:
        st.subheader("Distribuição Nacional por Tecnologia")

        col1, col2 = st.columns([1, 1])

        with col1:
            # Gráfico de pizza
            tech_values = [total_2g, total_3g, total_4g, total_5g]
            tech_labels = ['2G', '3G', '4G', '5G']
            fig = create_pie_chart(tech_values, tech_labels, "Distribuição de ERBs por Tecnologia")
            st.pyplot(fig)

        with col2:
            # Insights
            st.markdown("### 📋 Principais Insights")

            percentual_5g = (total_5g / total_geral) * 100
            percentual_4g = (total_4g / total_geral) * 100
            razao_5g_4g = (total_5g / total_4g) * 100 if total_4g > 0 else 0

            st.info(f"🚀 **5G representa {percentual_5g:.1f}% do total** de ERBs no Brasil")
            st.info(f"📡 **4G representa {percentual_4g:.1f}% do total** de ERBs no Brasil")
            st.info(f"⚡ **Razão 5G/4G: {razao_5g_4g:.1f}%** - indicador de modernização")

            # Operadora líder em 5G
            operadora_lider_5g = df_br.loc[df_br['5G'].idxmax(), 'Operadora']
            erbs_lider_5g = df_br['5G'].max()
            st.success(f"🏆 **{operadora_lider_5g}** lidera em 5G com {erbs_lider_5g:,.0f} ERBs")

# PÁGINA POR ESTADO
elif page == "🗺️ Por Estado":
    st.header("🗺️ Análise por Estado")

    # Filtro de estado
    estados = ["Todos"] + sorted(df_uf['UF'].unique().tolist())
    estado_selecionado = st.selectbox("Selecione um estado:", estados)

    if estado_selecionado == "Todos":
        # Análise de todos os estados
        df_estado_total = df_uf.groupby('UF')[['2G', '3G', '4G', '5G']].sum().reset_index()
        df_estado_total['Total'] = df_estado_total[['2G', '3G', '4G', '5G']].sum(axis=1)
        df_estado_total = df_estado_total.sort_values('Total', ascending=False)

        st.subheader("🏆 Ranking de Estados por Total de ERBs")

        # Top 10 estados
        top_10 = df_estado_total.head(10)

        fig, ax = plt.subplots(figsize=(12, 8))
        bars = ax.barh(top_10['UF'], top_10['Total'], color='#1f4e79')

        # Adicionar valores nas barras
        for i, bar in enumerate(bars):
            width = bar.get_width()
            ax.text(width + width*0.01, bar.get_y() + bar.get_height()/2,
                   f'{width:,.0f}'.replace(',', '.'),
                   ha='left', va='center', fontweight='bold')

        ax.set_xlabel('Total de ERBs', fontsize=12, fontweight='bold')
        ax.set_ylabel('Estado (UF)', fontsize=12, fontweight='bold')
        ax.set_title('Top 10 Estados por Total de ERBs', fontsize=14, fontweight='bold')
        ax.grid(axis='x', alpha=0.3)

        st.pyplot(fig)

        # Tabela completa
        st.subheader("📊 Dados Completos por Estado")
        st.dataframe(
            df_estado_total.style.format({
                '2G': '{:,.0f}',
                '3G': '{:,.0f}',
                '4G': '{:,.0f}',
                '5G': '{:,.0f}',
                'Total': '{:,.0f}'
            }),
            use_container_width=True,
            hide_index=True
        )

    else:
        # Análise do estado específico
        df_filtrado = df_uf[df_uf['UF'] == estado_selecionado]

        if df_filtrado.empty:
            st.warning(f"Não há dados disponíveis para {estado_selecionado}")
        else:
            # Métricas do estado
            total_estado = df_filtrado[['2G', '3G', '4G', '5G']].sum()

            col1, col2, col3, col4 = st.columns(4)
            with col1:
                st.metric("📱 2G", f"{total_estado['2G']:,.0f}".replace(",", "."))
            with col2:
                st.metric("📶 3G", f"{total_estado['3G']:,.0f}".replace(",", "."))
            with col3:
                st.metric("📡 4G", f"{total_estado['4G']:,.0f}".replace(",", "."))
            with col4:
                st.metric("🚀 5G", f"{total_estado['5G']:,.0f}".replace(",", "."))

            # Gráfico por operadora no estado
            st.subheader(f"📊 ERBs por Operadora em {estado_selecionado}")
            chart = create_bar_chart(df_filtrado, f"ERBs por Operadora - {estado_selecionado}")
            st.altair_chart(chart, use_container_width=True)

            # Tabela de dados
            st.subheader("📋 Dados Detalhados")
            df_display = df_filtrado[['Operadora', '2G', '3G', '4G', '5G']].copy()
            df_display['Total'] = df_display[['2G', '3G', '4G', '5G']].sum(axis=1)
            df_display = df_display.sort_values('Total', ascending=False)

            st.dataframe(
                df_display.style.format({
                    '2G': '{:,.0f}',
                    '3G': '{:,.0f}',
                    '4G': '{:,.0f}',
                    '5G': '{:,.0f}',
                    'Total': '{:,.0f}'
                }),
                use_container_width=True,
                hide_index=True
            )

# PÁGINA POR CIDADE
elif page == "🏙️ Por Cidade":
    st.header("🏙️ Análise por Cidade")

    # Filtros
    col1, col2 = st.columns(2)

    with col1:
        estados_cidade = ["Todos"] + sorted(df_cidade['UF'].unique().tolist())
        estado_sel = st.selectbox("Estado:", estados_cidade)

    with col2:
        if estado_sel != "Todos":
            cidades_filtradas = sorted(df_cidade[df_cidade['UF'] == estado_sel]['Cidade'].unique())
        else:
            cidades_filtradas = sorted(df_cidade['Cidade'].unique())

        cidades = ["Todas"] + cidades_filtradas
        cidade_sel = st.selectbox("Cidade:", cidades)

    # Aplicar filtros
    df_filtrado_cidade = df_cidade.copy()
    if estado_sel != "Todos":
        df_filtrado_cidade = df_filtrado_cidade[df_filtrado_cidade['UF'] == estado_sel]
    if cidade_sel != "Todas":
        df_filtrado_cidade = df_filtrado_cidade[df_filtrado_cidade['Cidade'] == cidade_sel]

    if cidade_sel != "Todas":
        # Análise cidade específica
        st.subheader(f"📊 Análise de {cidade_sel} - {estado_sel}")

        if df_filtrado_cidade.empty:
            st.warning("Não há dados para esta cidade")
        else:
            # Verificar se é capital
            if 'CAPITAL' in df_filtrado_cidade.columns:
                is_capital = df_filtrado_cidade['CAPITAL'].iloc[0] == "SIM"
                if is_capital:
                    st.info(f"🏛️ {cidade_sel} é uma capital estadual")

            # Métricas da cidade
            total_cidade = df_filtrado_cidade[['2G', '3G', '4G', '5G']].sum()

            col1, col2, col3, col4 = st.columns(4)
            with col1:
                st.metric("📱 2G", f"{total_cidade['2G']:,.0f}".replace(",", "."))
            with col2:
                st.metric("📶 3G", f"{total_cidade['3G']:,.0f}".replace(",", "."))
            with col3:
                st.metric("📡 4G", f"{total_cidade['4G']:,.0f}".replace(",", "."))
            with col4:
                st.metric("🚀 5G", f"{total_cidade['5G']:,.0f}".replace(",", "."))

            # Gráfico e tabela
            chart = create_bar_chart(df_filtrado_cidade, f"ERBs por Operadora - {cidade_sel}")
            st.altair_chart(chart, use_container_width=True)

            # Dados detalhados
            df_display = df_filtrado_cidade[['Operadora', '2G', '3G', '4G', '5G']].copy()
            df_display['Total'] = df_display[['2G', '3G', '4G', '5G']].sum(axis=1)
            df_display = df_display.sort_values('Total', ascending=False)

            st.dataframe(
                df_display.style.format({
                    '2G': '{:,.0f}', '3G': '{:,.0f}', '4G': '{:,.0f}', '5G': '{:,.0f}', 'Total': '{:,.0f}'
                }),
                use_container_width=True, hide_index=True
            )

    else:
        # Análise de múltiplas cidades
        if estado_sel != "Todos":
            st.subheader(f"🏙️ Principais Cidades de {estado_sel}")

            # Agrupar por cidade
            df_cidades_estado = df_filtrado_cidade.groupby('Cidade')[['2G', '3G', '4G', '5G']].sum().reset_index()
            df_cidades_estado['Total'] = df_cidades_estado[['2G', '3G', '4G', '5G']].sum(axis=1)
            df_cidades_estado = df_cidades_estado.sort_values('Total', ascending=False)

            # Top 15 cidades
            top_cidades = df_cidades_estado.head(15)

            fig, ax = plt.subplots(figsize=(12, 8))
            bars = ax.barh(top_cidades['Cidade'], top_cidades['Total'], color='#2E8B57')

            for i, bar in enumerate(bars):
                width = bar.get_width()
                ax.text(width + width*0.01, bar.get_y() + bar.get_height()/2,
                       f'{width:,.0f}'.replace(',', '.'),
                       ha='left', va='center', fontweight='bold')

            ax.set_xlabel('Total de ERBs', fontsize=12, fontweight='bold')
            ax.set_ylabel('Cidade', fontsize=12, fontweight='bold')
            ax.set_title(f'Top 15 Cidades por Total de ERBs - {estado_sel}', fontsize=14, fontweight='bold')
            ax.grid(axis='x', alpha=0.3)

            st.pyplot(fig)

        else:
            st.subheader("🏛️ Principais Capitais do Brasil")

            # Filtrar apenas capitais
            if 'CAPITAL' in df_cidade.columns:
                capitais = df_cidade[df_cidade['CAPITAL'] == "SIM"]
                df_capitais = capitais.groupby(['UF', 'Cidade'])[['2G', '3G', '4G', '5G']].sum().reset_index()
                df_capitais['Total'] = df_capitais[['2G', '3G', '4G', '5G']].sum(axis=1)
                df_capitais = df_capitais.sort_values('Total', ascending=False)

                # Top 15 capitais
                top_capitais = df_capitais.head(15)

                fig, ax = plt.subplots(figsize=(14, 8))
                labels = top_capitais['Cidade'] + ' (' + top_capitais['UF'] + ')'
                bars = ax.barh(labels, top_capitais['Total'], color='#8E44AD')

                for i, bar in enumerate(bars):
                    width = bar.get_width()
                    ax.text(width + width*0.01, bar.get_y() + bar.get_height()/2,
                           f'{width:,.0f}'.replace(',', '.'),
                           ha='left', va='center', fontweight='bold')

                ax.set_xlabel('Total de ERBs', fontsize=12, fontweight='bold')
                ax.set_ylabel('Capital', fontsize=12, fontweight='bold')
                ax.set_title('Top 15 Capitais por Total de ERBs', fontsize=14, fontweight='bold')
                ax.grid(axis='x', alpha=0.3)

                st.pyplot(fig)

# PÁGINA DE COMPARAÇÕES
elif page == "📈 Comparações":
    st.header("📈 Análises Comparativas")

    # Seletor de tipo de análise
    tipo_analise = st.selectbox(
        "Tipo de Análise:",
        ["🚀 Modernização 5G por Operadora", "🏆 Ranking por Tecnologia", "📊 Distribuição Regional"]
    )

    if tipo_analise == "🚀 Modernização 5G por Operadora":
        st.subheader("Análise de Modernização 5G")

        # Calcular razão 5G/4G por operadora
        df_modernizacao = df_br.copy()
        df_modernizacao['Total'] = df_modernizacao[['2G', '3G', '4G', '5G']].sum(axis=1)
        df_modernizacao['Razão_5G_4G'] = (df_modernizacao['5G'] / df_modernizacao['4G'] * 100).round(2)
        df_modernizacao['Percentual_5G'] = (df_modernizacao['5G'] / df_modernizacao['Total'] * 100).round(2)

        # Filtrar operadoras principais (com mais de 1000 ERBs)
        df_principais = df_modernizacao[df_modernizacao['Total'] >= 1000].sort_values('Total', ascending=False)

        if not df_principais.empty:
            # Gráfico de dispersão: Volume vs Modernização
            fig, ax = plt.subplots(figsize=(12, 8))

            scatter = ax.scatter(
                df_principais['Total'],
                df_principais['Razão_5G_4G'],
                s=df_principais['5G']/df_principais['5G'].max() * 500,
                c=df_principais['Percentual_5G'],
                cmap='viridis',
                alpha=0.7,
                edgecolors='black',
                linewidth=1
            )

            # Adicionar rótulos
            for i, row in df_principais.iterrows():
                ax.annotate(
                    row['Operadora'],
                    (row['Total'], row['Razão_5G_4G']),
                    xytext=(5, 5),
                    textcoords='offset points',
                    fontweight='bold',
                    fontsize=10
                )

            ax.set_xlabel('Total de ERBs', fontsize=12, fontweight='bold')
            ax.set_ylabel('Razão 5G/4G (%)', fontsize=12, fontweight='bold')
            ax.set_title('Modernização 5G: Volume vs Eficiência', fontsize=14, fontweight='bold')
            ax.grid(True, alpha=0.3)

            # Colorbar
            cbar = plt.colorbar(scatter)
            cbar.set_label('Percentual de 5G (%)', fontweight='bold')

            st.pyplot(fig)

            # Tabela de dados
            st.subheader("📊 Dados de Modernização")
            df_display = df_principais[['Operadora', 'Total', '4G', '5G', 'Razão_5G_4G', 'Percentual_5G']].copy()

            st.dataframe(
                df_display.style.format({
                    'Total': '{:,.0f}',
                    '4G': '{:,.0f}',
                    '5G': '{:,.0f}',
                    'Razão_5G_4G': '{:.2f}%',
                    'Percentual_5G': '{:.2f}%'
                }),
                use_container_width=True,
                hide_index=True
            )

    elif tipo_analise == "🏆 Ranking por Tecnologia":
        st.subheader("Ranking de Operadoras por Tecnologia")

        tecnologia = st.selectbox("Selecione a tecnologia:", ['2G', '3G', '4G', '5G'])

        df_ranking = df_br.sort_values(tecnologia, ascending=False)

        # Gráfico de barras horizontal
        fig, ax = plt.subplots(figsize=(10, 8))
        colors = {'2G': '#FF6B35', '3G': '#004E89', '4G': '#1A8F5A', '5G': '#8E44AD'}

        bars = ax.barh(df_ranking['Operadora'], df_ranking[tecnologia], color=colors[tecnologia])

        for i, bar in enumerate(bars):
            width = bar.get_width()
            ax.text(width + width*0.01, bar.get_y() + bar.get_height()/2,
                   f'{width:,.0f}'.replace(',', '.'),
                   ha='left', va='center', fontweight='bold')

        ax.set_xlabel(f'Número de ERBs {tecnologia}', fontsize=12, fontweight='bold')
        ax.set_ylabel('Operadora', fontsize=12, fontweight='bold')
        ax.set_title(f'Ranking de Operadoras - Tecnologia {tecnologia}', fontsize=14, fontweight='bold')
        ax.grid(axis='x', alpha=0.3)

        st.pyplot(fig)

        # Estatísticas
        total_tech = df_ranking[tecnologia].sum()
        lider = df_ranking.iloc[0]

        col1, col2, col3 = st.columns(3)
        with col1:
            st.metric(f"Total {tecnologia} Nacional", f"{total_tech:,.0f}".replace(",", "."))
        with col2:
            st.metric("Operadora Líder", lider['Operadora'])
        with col3:
            participacao = (lider[tecnologia] / total_tech * 100)
            st.metric("Participação do Líder", f"{participacao:.1f}%")

    else:  # Distribuição Regional
        st.subheader("📊 Distribuição Regional de ERBs")

        # Definir regiões
        regioes = {
            'Norte': ['AC', 'AM', 'AP', 'PA', 'RO', 'RR', 'TO'],
            'Nordeste': ['AL', 'BA', 'CE', 'MA', 'PB', 'PE', 'PI', 'RN', 'SE'],
            'Centro-Oeste': ['DF', 'GO', 'MT', 'MS'],
            'Sudeste': ['ES', 'MG', 'RJ', 'SP'],
            'Sul': ['PR', 'RS', 'SC']
        }

        # Calcular totais por região
        df_regional = []
        for regiao, ufs in regioes.items():
            df_regiao = df_uf[df_uf['UF'].isin(ufs)]
            totais = df_regiao[['2G', '3G', '4G', '5G']].sum()
            df_regional.append({
                'Região': regiao,
                '2G': totais['2G'],
                '3G': totais['3G'],
                '4G': totais['4G'],
                '5G': totais['5G'],
                'Total': totais.sum()
            })

        df_regional = pd.DataFrame(df_regional)
        df_regional = df_regional.sort_values('Total', ascending=False)

        # Gráfico de barras empilhadas
        fig, ax = plt.subplots(figsize=(12, 8))

        bottom = np.zeros(len(df_regional))
        colors = ['#FF6B35', '#004E89', '#1A8F5A', '#8E44AD']
        techs = ['2G', '3G', '4G', '5G']

        for i, tech in enumerate(techs):
            ax.bar(df_regional['Região'], df_regional[tech], bottom=bottom,
                   label=tech, color=colors[i], alpha=0.8)
            bottom += df_regional[tech]

        ax.set_xlabel('Região', fontsize=12, fontweight='bold')
        ax.set_ylabel('Número de ERBs', fontsize=12, fontweight='bold')
        ax.set_title('Distribuição de ERBs por Região e Tecnologia', fontsize=14, fontweight='bold')
        ax.legend(loc='upper right')
        ax.grid(axis='y', alpha=0.3)

        # Adicionar valores totais no topo
        for i, row in df_regional.iterrows():
            ax.text(i, row['Total'] + row['Total']*0.01,
                   f"{row['Total']:,.0f}".replace(',', '.'),
                   ha='center', va='bottom', fontweight='bold')

        st.pyplot(fig)

        # Tabela regional
        st.subheader("📋 Dados Regionais Detalhados")
        st.dataframe(
            df_regional.style.format({
                '2G': '{:,.0f}',
                '3G': '{:,.0f}',
                '4G': '{:,.0f}',
                '5G': '{:,.0f}',
                'Total': '{:,.0f}'
            }),
            use_container_width=True,
            hide_index=True
        )

# Footer
st.markdown("---")
st.markdown("""
<div style='text-align: center; color: #666; padding: 20px;'>
    <p><strong>🏢 Dashboard ANATEL - Telecomunicações Brasil</strong></p>
    <p>Dados: Agência Nacional de Telecomunicações (ANATEL)</p>
    <p>Desenvolvido com ❤️ usando Streamlit</p>
</div>
""", unsafe_allow_html=True)

Writing streamlit_app.py


In [4]:
import subprocess
import threading
import time

def run_app():
    """Executa o Streamlit"""
    subprocess.run([
        "streamlit", "run", "streamlit_app.py",
        "--server.port=8501",
        "--server.headless=true",
        "--server.enableCORS=false",
        "--server.enableXsrfProtection=false"
    ])

# Iniciar Streamlit em background
print("🚀 Iniciando Dashboard ANATEL...")
app_thread = threading.Thread(target=run_app)
app_thread.daemon = True
app_thread.start()

# Aguardar inicialização
time.sleep(5)

# Configurar túnel público
public_url = ngrok.connect(8501)

print("=" * 70)
print("🎉 DASHBOARD ANATEL ESTÁ ONLINE!")
print("=" * 70)
print(f"🌐 URL Pública: {public_url}")
print("=" * 70)
print("📱 Acesse a URL no navegador para usar o dashboard")
print("🔄 Para parar: Runtime → Interrupt execution")
print("📊 Funcionalidades:")
print("   • Análise Nacional de ERBs")
print("   • Análise por Estado")
print("   • Análise por Cidade")
print("   • Comparações e Rankings")
print("=" * 70)

# Manter rodando
try:
    while True:
        time.sleep(60)
        print("⚡ Dashboard rodando... Acesse a URL acima!")
except KeyboardInterrupt:
    print("\n🛑 Parando dashboard...")
    ngrok.disconnect(public_url)
    print("✅ Dashboard encerrado!")

🚀 Iniciando Dashboard ANATEL...
🎉 DASHBOARD ANATEL ESTÁ ONLINE!
🌐 URL Pública: NgrokTunnel: "https://a2b47007c0a0.ngrok-free.app" -> "http://localhost:8501"
📱 Acesse a URL no navegador para usar o dashboard
🔄 Para parar: Runtime → Interrupt execution
📊 Funcionalidades:
   • Análise Nacional de ERBs
   • Análise por Estado
   • Análise por Cidade
   • Comparações e Rankings
⚡ Dashboard rodando... Acesse a URL acima!

🛑 Parando dashboard...
✅ Dashboard encerrado!
